In [8]:
from adapter.proceed import Proceed
from tsfm_public.models.tinytimemixer import TinyTimeMixerConfig, TinyTimeMixerForPrediction

In [19]:
from dataclasses import dataclass

@dataclass
class Args:
    freeze: bool = False
    concept_dim: int = 200
    bottleneck_dim: int = 32
    seq_len: int = 512
    pred_len: int = 96
    enc_in: int = 7
    ema: int = 0
    act: str = 'sigmoid'
    individual_generator: bool = False
    wo_clip: bool = False
    do_predict: bool = True
    tune_mode: str = 'down_up'
    merge_weights: int = 1

In [20]:
import torch
import torch.nn as nn
import transformers
from adapter.module.generator import AdaptGenerator
from adapter.module import down_up


def normalize(W, max_norm=1):
    W_norm = torch.norm(W, dim=-1, keepdim=True)
    scale = torch.clip(max_norm / W_norm, max=1)
    return W * scale


class Transpose(nn.Module):
    def __init__(self, *dims, contiguous=False):
        super().__init__()
        self.dims, self.contiguous = dims, contiguous
    def forward(self, x):
        if self.contiguous: return x.transpose(*self.dims).contiguous()
        else: return x.transpose(*self.dims)


class Proceed(nn.Module):
    def __init__(self, backbone, args):
        super().__init__()
        self.args = args
        if args.freeze:
            backbone.requires_grad_(False)
        self.backbone = add_adapters_(backbone, args)
        self.more_bias = not args.freeze
        self.generator = AdaptGenerator(backbone, args.concept_dim,
                                        activation=nn.Sigmoid if args.act == 'sigmoid' else nn.Identity,
                                        adaptive_dim=False, need_bias=self.more_bias,
                                        shared=not args.individual_generator,
                                        mid_dim=args.bottleneck_dim)
        # print(self.adapters)
        self.register_buffer('recent_batch', torch.zeros(1, args.seq_len + args.pred_len, args.enc_in), persistent=False)
        if args.ema > 0:
            self.register_buffer('recent_concept', None, persistent=True)
        self.mlp1 = nn.Sequential(Transpose(-1, -2), nn.Linear(args.seq_len, args.concept_dim), nn.GELU(),
                                  nn.Linear(args.concept_dim, args.concept_dim))
        self.mlp2 = nn.Sequential(Transpose(-1, -2), nn.Linear(args.seq_len + args.pred_len, args.concept_dim), nn.GELU(),
                                  nn.Linear(args.concept_dim, args.concept_dim))
        self.ema = args.ema
        self.flag_online_learning = False
        self.flag_update = False
        self.flag_current = False
        self.flag_basic = False

    def generate_adaptation(self, x):
        concept = self.mlp1(x).mean(-2)
        recent_concept = self.mlp2(self.recent_batch).mean(-2).mean(list(range(0, self.recent_batch.dim() - 2)))
        if self.ema > 0:
            if self.recent_concept is not None:
                recent_concept = self.recent_concept * self.ema + recent_concept * (1 - self.ema)
            if self.flag_update or self.flag_online_learning and not self.flag_current:
                self.recent_concept = recent_concept.detach()
        drift = concept - recent_concept
        res = self.generator(drift, need_clip=not self.args.wo_clip)
        return res

    def forward(self, *x):
        if self.flag_basic:
            adaptations = {}
            for i, (k, adapter) in enumerate(self.generator.bottlenecks.items()):
                adaptations[k] = adapter.biases[-1] if adapter.need_bias else [None] * len(self.generator.dim_name_dict[k])
        else:
            adaptations = self.generate_adaptation(x[0])
        for out_dim, adaptation in adaptations.items():
            for i in range(len(adaptation)):
                name = self.generator.dim_name_dict[out_dim][i]
                self.backbone.get_submodule(name).assign_adaptation(adaptation[i])
        if self.args.do_predict:
            return self.backbone(*x)
        else:
            return self.backbone(*x)

    def freeze_adapter(self, freeze=True):
        for module_name in ['mlp1', 'mlp2']:
            if hasattr(self, module_name):
                getattr(self, module_name).requires_grad_(not freeze)
                getattr(self, module_name).zero_grad(set_to_none=True)
        for adapter in self.generator.bottlenecks.values():
            adapter.weights.requires_grad_(not freeze)
            adapter.weights.zero_grad(set_to_none=True)
            adapter.biases[:len(adapter.weights) - 1].requires_grad_(not freeze)
            adapter.biases[:len(adapter.weights) - 1].zero_grad(set_to_none=True)

    def freeze_bias(self, freeze=True):
        if self.more_bias:
            for adapter in self.generator.bottlenecks.values():
                adapter.biases[-1].requires_grad_(not freeze)
                adapter.biases[-1:].zero_grad(set_to_none=True)


def add_adapters_(parent_module: nn.Module, args, top_level=True):
    for name, module in parent_module.named_children():
        if args.tune_mode == 'all_down_up' and isinstance(module, (nn.Conv1d, nn.Linear, transformers.Conv1D,
                                                                     nn.LayerNorm, nn.BatchNorm1d)):
            down_up.add_down_up_(parent_module, name, freeze_weight=args.freeze, merge_weights=args.merge_weights,)
        elif args.tune_mode == 'down_up' and isinstance(module, (nn.Conv1d, nn.Linear, transformers.Conv1D)):
            down_up.add_down_up_(parent_module, name, freeze_weight=args.freeze, merge_weights=args.merge_weights,)
        else:
            add_adapters_(module, args, False)
    return parent_module


In [26]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.backbone = TinyTimeMixerForPrediction.from_pretrained('ibm-granite/granite-timeseries-ttm-r1')

    def forward(self, x, x_mark=None, return_emb=False):
        outputs = self.backbone(x, return_loss=False, return_dict=True)
        if hasattr(outputs, 'prediction_outputs'):
            return outputs.prediction_outputs
        if isinstance(outputs, tuple) and len(outputs) > 0:
            return outputs[0]
        return outputs

In [30]:
ttm_backbone = Model()

In [31]:
model = Proceed(ttm_backbone, Args)

In [48]:
seqs = torch.rand(4, 512, 7) # [Batch size, sequence length, dimension]
mean, std = seqs.mean(dim=1, keepdim=True), seqs.std(dim=1, keepdim=True)

normed_seqs = (seqs - mean) / std

In [49]:
normed_res = model(normed_seqs)

In [50]:
res = normed_res * std + mean
res

tensor([[[0.5375, 0.5664, 0.4768,  ..., 0.3295, 0.5498, 0.5852],
         [0.5391, 0.4717, 0.5209,  ..., 0.4145, 0.5440, 0.5514],
         [0.5244, 0.4561, 0.5477,  ..., 0.4314, 0.5267, 0.5401],
         ...,
         [0.4964, 0.4695, 0.4715,  ..., 0.4657, 0.5087, 0.5057],
         [0.5084, 0.4649, 0.4908,  ..., 0.4813, 0.5241, 0.5128],
         [0.5337, 0.4590, 0.4922,  ..., 0.4890, 0.5432, 0.5010]],

        [[0.5297, 0.3159, 0.4283,  ..., 0.5971, 0.4276, 0.3825],
         [0.5241, 0.4097, 0.4769,  ..., 0.5534, 0.4748, 0.3910],
         [0.5298, 0.4482, 0.4958,  ..., 0.5376, 0.5194, 0.4247],
         ...,
         [0.5346, 0.4758, 0.4899,  ..., 0.5418, 0.5052, 0.4785],
         [0.5310, 0.4840, 0.4956,  ..., 0.5329, 0.5055, 0.4701],
         [0.5485, 0.4682, 0.4982,  ..., 0.5250, 0.5159, 0.4736]],

        [[0.5652, 0.5755, 0.4688,  ..., 0.6032, 0.4685, 0.6126],
         [0.5294, 0.5535, 0.4655,  ..., 0.5568, 0.4944, 0.5545],
         [0.5170, 0.5324, 0.4760,  ..., 0.5186, 0.4774, 0.